# NFL Franchise Valuation — Machine Learning
**Private Equity Data Science Project**

In [6]:
import pandas as pd
import numpy as np
import requests
import time
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## 1. Load Data

In [7]:
HEADERS = {'User-Agent': 'Mozilla/5.0'}

def fetch(url):
    """Fetch page HTML with a browser User-Agent to avoid 403 errors."""
    return requests.get(url, headers=HEADERS).content

html = fetch('https://en.wikipedia.org/wiki/Forbes_list_of_the_most_valuable_NFL_teams')

# Table - current rankings: Rank | Swing | Team | State | Value | Change | Revenue | Operating income
val = pd.read_html(html)[0][['Team', 'Value', 'Revenue', 'Operating income']].copy()

# convert all to $M
def to_millions(series):
    s = series.astype(str)
    billions = s.str.contains('billion', case=False, na=False)
    millions = s.str.contains('million', case=False, na=False)
    s = pd.to_numeric(s.str.replace(r'[^0-9.]', '', regex=True), errors='coerce')
    return np.where(billions, s * 1000, np.where(millions, s, np.nan))

for col in ['Value', 'Revenue', 'Operating income']:
    val[col] = to_millions(val[col])

val.columns = ['team', 'valuation_mm', 'revenue_mm', 'op_income_mm']
val = val.dropna()
print(val.to_string(index=False))

                 team  valuation_mm  revenue_mm  op_income_mm
       Dallas Cowboys       13000.0      1234.0         629.0
     Los Angeles Rams       10500.0       764.0         244.0
      New York Giants       10100.0       707.0         181.0
 New England Patriots        9000.0       762.0         222.0
  San Francisco 49ers        8600.0       723.0         115.0
  Philadelphia Eagles        8300.0       688.0         117.0
        Chicago Bears        8200.0       629.0          80.0
        New York Jets        8100.0       663.0         180.0
    Las Vegas Raiders        7700.0       832.0         179.0
Washington Commanders        7600.0       644.0         116.0
       Miami Dolphins        7500.0       656.0          63.0
       Houston Texans        7400.0       687.0         156.0
       Denver Broncos        6800.0       645.0         103.0
     Seattle Seahawks        6700.0       624.0         143.0
    Green Bay Packers        6650.0       719.0          83.0
 Tampa B

In [8]:
#2. NFLVERSE: regular-season win - 2019-2025
games_raw = pd.read_csv(
    'https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv',
    low_memory=False
)

ABBREV = {
    'ARI':'Arizona Cardinals',    'ATL':'Atlanta Falcons',
    'BAL':'Baltimore Ravens',     'BUF':'Buffalo Bills',
    'CAR':'Carolina Panthers',    'CHI':'Chicago Bears',
    'CIN':'Cincinnati Bengals',   'CLE':'Cleveland Browns',
    'DAL':'Dallas Cowboys',       'DEN':'Denver Broncos',
    'DET':'Detroit Lions',        'GB' :'Green Bay Packers',
    'HOU':'Houston Texans',       'IND':'Indianapolis Colts',
    'JAX':'Jacksonville Jaguars', 'KC' :'Kansas City Chiefs',
    'LA' :'Los Angeles Rams',     'LAC':'Los Angeles Chargers',
    'LV' :'Las Vegas Raiders',    'MIA':'Miami Dolphins',
    'MIN':'Minnesota Vikings',    'NE' :'New England Patriots',
    'NO' :'New Orleans Saints',   'NYG':'New York Giants',
    'NYJ':'New York Jets',        'PHI':'Philadelphia Eagles',
    'PIT':'Pittsburgh Steelers',  'SEA':'Seattle Seahawks',
    'SF' :'San Francisco 49ers',  'TB' :'Tampa Bay Buccaneers',
    'TEN':'Tennessee Titans',     'WAS':'Washington Commanders',
}

games_reg = games_raw[
    (games_raw['season'].between(2019, 2025)) & (games_raw['game_type'] == 'REG')].copy()

home = games_reg[['home_team','result']].rename(columns={'home_team':'team'})
away = games_reg[['away_team','result']].rename(columns={'away_team':'team'})
away['result'] = -away['result']
all_games        = pd.concat([home, away])
all_games['team']= all_games['team'].replace('OAK','LV')   # Raiders relocation fix
all_games['win'] = (all_games['result'] > 0).astype(int)
all_games['tie'] = (all_games['result'] == 0).astype(int)

win_pct = (all_games.groupby('team').agg(games=('win','count'), wins=('win','sum'), ties=('tie','sum'))
    .reset_index()
)
win_pct['win_pct_5yr'] = (win_pct['wins'] + 0.5 * win_pct['ties']) / win_pct['games']
win_pct['team']        = win_pct['team'].map(ABBREV)
win_pct                = win_pct[['team','win_pct_5yr']].dropna(subset=['team'])
print('\n── Win % 2019-2025 ──')
print(win_pct.sort_values('win_pct_5yr', ascending=False).to_string(index=False))

#3. NFLVERSE: playoff appearances 2019-2025
games_po = games_raw[
    (games_raw['season'].between(2019, 2025)) &
    (games_raw['game_type'].isin(['WC','DIV','CON','SB']))
].copy()

home_po = games_po[['home_team']].rename(columns={'home_team':'team'})
away_po = games_po[['away_team']].rename(columns={'away_team':'team'})
playoff_counts = (pd.concat([home_po, away_po]).assign(team=lambda d: d['team'].replace('OAK','LV').map(ABBREV))
    .dropna(subset=['team'])
    .groupby('team')
    .size()
    .reset_index(name='playoff_games_5yr')
)
print('\n── Playoff appearances 2019-2025 ──')
print(playoff_counts.sort_values('playoff_games_5yr', ascending=False).to_string(index=False))


── Win % 2019-2025 ──
                 team  win_pct_5yr
   Kansas City Chiefs     0.717949
        Buffalo Bills     0.715517
    Green Bay Packers     0.653846
     Baltimore Ravens     0.649573
  Philadelphia Eagles     0.619658
     Seattle Seahawks     0.615385
  San Francisco 49ers     0.615385
  Pittsburgh Steelers     0.585470
    Minnesota Vikings     0.581197
     Los Angeles Rams     0.581197
 Tampa Bay Buccaneers     0.564103
       Dallas Cowboys     0.551282
   New Orleans Saints     0.521368
       Miami Dolphins     0.504274
 New England Patriots     0.504274
 Los Angeles Chargers     0.495726
        Detroit Lions     0.487179
   Indianapolis Colts     0.482906
       Denver Broncos     0.478632
       Houston Texans     0.457265
   Cincinnati Bengals     0.452586
     Cleveland Browns     0.435897
     Tennessee Titans     0.435897
        Chicago Bears     0.410256
      Atlanta Falcons     0.410256
Washington Commanders     0.397436
    Las Vegas Raiders     0.3931

In [9]:
#4. HARDCODED: attendance, market, SB wins, stadium, social, TV market, rev_2020 ──
extra = pd.DataFrame({
    'team': [
        'Dallas Cowboys','Los Angeles Rams','New England Patriots','New York Giants',
        'Las Vegas Raiders','San Francisco 49ers','New York Jets','Miami Dolphins',
        'Philadelphia Eagles','Washington Commanders','Chicago Bears','Denver Broncos',
        'Seattle Seahawks','Kansas City Chiefs','Pittsburgh Steelers','Green Bay Packers',
        'Baltimore Ravens','Los Angeles Chargers','New Orleans Saints','Detroit Lions',
        'Atlanta Falcons','Carolina Panthers','Cincinnati Bengals','Buffalo Bills',
        'Houston Texans','Minnesota Vikings','Indianapolis Colts','Tennessee Titans',
        'Jacksonville Jaguars','Arizona Cardinals','Cleveland Browns','Tampa Bay Buccaneers',
    ],
    # ESPN reported avg
    'avg_attendance':   [92972,68411,65878,82500,63704,68469,80203,65095,69738,67717,
                         58649,75858,68961,76285,68400,77788,71008,65267,73051,64873,
                         70738,73892,65267,71551,71819,66441,62243,68983,69021,63211,67895,65641],
    # Census metro area population (millions)
    'market_mm':        [7.8,13.2,4.9,19.8,2.2,4.7,19.8,6.1,6.2,6.4,9.5,2.9,
                         4.0,2.2,2.4,0.3,2.9,13.2,1.3,4.4,6.2,2.7,2.3,1.2,
                         7.3,3.7,2.1,2.0,1.6,5.0,2.1,3.2],
    # All-time Super Bowl wins
    'sb_wins':          [5,2,6,4,1,5,0,2,1,0,1,3,1,4,6,4,2,0,1,0,0,0,0,0,0,1,2,0,0,1,0,1],
    # Stadium age in years
    'stadium_age':      [16,4,24,50,4,11,50,8,24,54,104,54,28,52,44,104,
                         27,4,57,23,8,25,34,2,23,10,36,54,32,18,5,22],
    # 1 = team/owner controls building economics; 0 = city/lease
    'stadium_owned':    [1,1,1,0,1,1,0,1,1,0,0,0,1,0,1,1,0,1,1,1,1,1,0,1,0,1,0,0,1,1,1,1],
    # Annual naming rights deal value ($M); 0 = no deal
    'naming_rights_mm': [17.0,0.0,8.0,0.0,25.0,0.0,0.0,0.0,8.5,0.0,0.0,0.0,
                         0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.5,0.0,0.0,0.0,17.6,
                         0.0,12.0,8.0,0.0,20.0,0.0,0.0,0.0],
    # Instagram + Twitter/X combined followers (~2025, millions)
    'social_following_mm': [35.2,12.1,14.8,10.2,10.5,12.3,8.4,10.8,13.1,5.2,9.8,10.1,
                            11.2,17.4,12.3,9.8,8.4,6.2,8.9,7.1,9.3,6.8,7.2,9.4,
                            8.7,9.1,7.3,6.2,5.8,6.1,8.2,9.8],
    # Nielsen DMA TV market rank (1 = largest; inverted in feature engineering)
    'tv_market_rank':   [5,2,7,1,40,6,1,16,4,8,3,17,12,31,23,69,
                         26,2,51,11,9,24,35,52,10,15,25,27,42,11,18,13],
    # Forbes reported revenue 2020 ($M) — used to compute 5yr CAGR
    'rev_2020_mm':      [1100,640,628,619,604,601,592,568,574,543,528,560,
                         529,504,524,508,520,481,501,504,519,480,463,462,
                         561,504,489,476,440,446,477,462],
})

In [10]:
#5. MERGE Data
df = (extra
    .merge(win_pct,        on='team', how='left')
    .merge(playoff_counts, on='team', how='left')
    .merge(val,            on='team', how='left')
)
df['playoff_games_5yr'] = df['playoff_games_5yr'].fillna(0)  # missed playoffs entirely

#  6. FEATURE ENGINEERING
df['ebitda_margin'] = df['op_income_mm']   / df['revenue_mm']
df['local_rev_mm'] = df['revenue_mm']      - 432.6          # strip avg national TV share
df['market_penetration'] = df['avg_attendance']  / (df['market_mm'] * 1e6)
df['tv_market_rank_inv'] = 1 / df['tv_market_rank']
df['rev_cagr_5yr']  = (df['revenue_mm'] / df['rev_2020_mm']) ** (1/5) - 1
df['op_leverage']         = df['revenue_mm']    / df['op_income_mm']   # efficiency ratio
df['rev_per_capita']      = df['revenue_mm']    / df['market_mm']
df['naming_rev_per_seat'] = df['naming_rights_mm'] / (df['avg_attendance'] * 8)  # 8 home games

## 2. Features & Target

In [11]:
features = [
    'revenue_mm',
    'op_income_mm',
    'rev_cagr_5yr',
    'social_following_mm',
    'tv_market_rank_inv',
    'win_pct_5yr',
    'stadium_owned',
    'avg_attendance',
]

df_model = df[features + ['valuation_mm','team']].dropna()
X = df_model[features]
y = df_model['valuation_mm']
print(f'\nSamples: {len(df_model)}')


Samples: 32


## 3. Train / Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Test teams: {df_model.loc[X_test.index, "team"].tolist()}')

Train: 22  |  Test: 10
Test teams: ['Arizona Cardinals', 'Green Bay Packers', 'Houston Texans', 'Los Angeles Chargers', 'Philadelphia Eagles', 'Washington Commanders', 'Cleveland Browns', 'Minnesota Vikings', 'Seattle Seahawks', 'Dallas Cowboys']


## 4. Train Models

In [13]:
# Using LOOCV for all three models — trains on 31, predicts the held-out 1, rotates.
loo  = LeaveOneOut()
models = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=50))]),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=2,
                                                   learning_rate=0.05, random_state=42),
}
print(f'\n{"Model":<22} {"MAE ($M)":>10} {"R²":>8}')
print('─' * 42)
results = {}
for name, model in models.items():
    preds = np.zeros(len(y))
    for train_idx, test_idx in loo.split(X):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds[test_idx] = model.predict(X.iloc[test_idx])
    mae = mean_absolute_error(y, preds)
    r2  = r2_score(y, preds)
    results[name] = {'preds': preds, 'mae': mae, 'r2': r2}   # store for best-model selection and prediction table
    print(f'{name:<22} ${mae:>8,.0f} {r2:>8.3f}')
    model.fit(X, y)

best_name  = min(results, key=lambda k: results[k]['mae'])
best_preds = results[best_name]['preds']
print(f'\nBest model: {best_name}\n')



Model                    MAE ($M)       R²
──────────────────────────────────────────
Ridge                  $     800    0.672
Random Forest          $     641    0.699
Gradient Boosting      $     749    0.655

Best model: Random Forest



# 5. Model Output vs Actual Results

In [14]:
print(f'{"Team":<25} {"Actual ($B)":>12} {"Predicted ($B)":>15} {"Error ($B)":>12}')
print('─' * 66)
for team, actual, pred in zip(df_model['team'], y.values, best_preds):
    error = pred - actual
    print(f'{team:<25} ${actual/1000:>10.2f}B  ${pred/1000:>12.2f}B  {error/1000:>+10.2f}B')


Team                       Actual ($B)  Predicted ($B)   Error ($B)
──────────────────────────────────────────────────────────────────
Dallas Cowboys            $     13.00B  $        9.61B       -3.39B
Los Angeles Rams          $     10.50B  $        8.92B       -1.58B
New England Patriots      $      9.00B  $        9.69B       +0.69B
New York Giants           $     10.10B  $        8.29B       -1.81B
Las Vegas Raiders         $      7.70B  $        7.96B       +0.26B
San Francisco 49ers       $      8.60B  $        7.67B       -0.93B
New York Jets             $      8.10B  $        8.70B       +0.60B
Miami Dolphins            $      7.50B  $        6.85B       -0.65B
Philadelphia Eagles       $      8.30B  $        8.02B       -0.28B
Washington Commanders     $      7.60B  $        7.20B       -0.40B
Chicago Bears             $      8.20B  $        7.21B       -0.99B
Denver Broncos            $      6.80B  $        7.17B       +0.37B
Seattle Seahawks          $      6.70B  $        

## 6. Feature Importance

In [15]:
#8. FEATURE IMPORTANCE (Random Forest)
rf_model   = models['Random Forest']
importance = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print('\nFeature Importance (Random Forest):')
print('─' * 40)
for feat, score in importance.items():
    bar = '█' * int(score * 50)
    print(f'  {feat:<22} {score:.3f}  {bar}')


Feature Importance (Random Forest):
────────────────────────────────────────
  op_income_mm           0.338  ████████████████
  revenue_mm             0.318  ███████████████
  tv_market_rank_inv     0.188  █████████
  avg_attendance         0.058  ██
  social_following_mm    0.048  ██
  rev_cagr_5yr           0.032  █
  win_pct_5yr            0.015  
  stadium_owned          0.002  
